# W&B Run 表复制工具

将源项目中某个 run 的指定"表"（即一组同前缀的 metric 列）的所有 epoch 数据，
复制到目标项目的目标 run 中，并可重命名表前缀。

**用法**：修改下方 `TASKS` 列表，每个任务是一个字典，指定源/目标的项目、run、表名。

需要在 `conda` 的 `zkj-work` 环境里运行。

In [1]:
import math
import sys
from dataclasses import dataclass, field
from typing import Optional

import pandas as pd
import wandb
from tqdm.auto import tqdm

print(sys.executable)
print('wandb', wandb.__version__)

/root/anaconda3/envs/zkj-work/bin/python
wandb 0.25.1


In [ ]:
ENTITY = 'kejian-zhao-tsinghua-university'

# ── 在这里定义复制任务 ──────────────────────────────────────────────
# 每个任务描述: 从哪个项目的哪个 run 读取哪张"表", 写到哪个项目的哪个 run 的哪张"表"。
#
# 字段说明:
#   src_project    : 源 wandb 项目名
#   src_run_name   : 源 run 名称 (如果有多个同名 run 会报错)
#   src_table      : 源表前缀, 比如 "summary/work_tpir_at_far_"
#                    会匹配 history 中所有以此开头的列
#   dst_project    : 目标 wandb 项目名
#   dst_run_name   : 目标 run 名称
#                    - 如果目标 run 已存在, 会 **追加写入** (resume)
#                    - 如果不存在, 会创建新 run
#   dst_table      : 目标表前缀, 比如 "summary/work_0213_tpir_at_far_"
#                    源表前缀会被替换为目标表前缀
#   extra_columns  : 额外要一起复制的列名列表 (如 step 指标、epoch 等)
#                    默认自动携带 step 指标列
#
# ── 示例 ──────────────────────────────────────────────────────────
# TASKS = [
#     {
#         'src_project': 'work_0213_eval_all_s2',
#         'src_run_name': 's3_0315',
#         'src_table': 'summary/work_tpir_at_far_',
#         'dst_project': 'eval_log',
#         'dst_run_name': 's3_0315',
#         'dst_table': 'summary/work_0213_tpir_at_far_',
#         'extra_columns': [],
#     },
#     # 可以继续添加更多任务...
# ]

In [9]:
# ── 核心工具函数 ────────────────────────────────────────────────────

STEP_METRIC_CANDIDATES = ('trainer/global_step', 'step', 'epoch')
DROP_PREFIXES = ('system/', '_')


def is_missing(value):
    """判断值是否为缺失值 (None / NaN)。"""
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    try:
        m = pd.isna(value)
    except (TypeError, ValueError):
        return False
    return bool(m) if isinstance(m, bool) else False


def find_run_by_name(api, project, run_name):
    """在项目中按名称精确查找唯一 run, 找不到或有重名则报错。"""
    path = f'{ENTITY}/{project}'
    matches = [r for r in api.runs(path) if r.name == run_name]
    if not matches:
        raise ValueError(f'在 {path} 中找不到 run: {run_name}')
    if len(matches) > 1:
        raise ValueError(
            f'在 {path} 中找到多个同名 run: {run_name}, '
            f'ids={[r.id for r in matches]}'
        )
    return matches[0]


def find_step_metric(columns):
    """从列名列表中找到第一个匹配的 step 指标。"""
    for c in STEP_METRIC_CANDIDATES:
        if c in columns:
            return c
    return None


def inspect_run(api, project, run_name):
    """
    诊断工具: 打印源 run 的 history 和 summary 中所有可用列。
    帮助确认 src_table 应该填什么前缀。
    """
    run = find_run_by_name(api, project, run_name)
    print(f'Run: {run.name} ({run.id})')
    print(f'URL: {run.url}')
    print()

    # ── history 列 ──
    rows = list(run.scan_history())
    if not rows:
        print('History: (empty)')
    else:
        df = pd.DataFrame(rows)
        # 过滤掉 system/ 和 _ 开头的列
        user_cols = [c for c in df.columns
                     if not any(c.startswith(p) for p in DROP_PREFIXES)]
        print(f'History: {len(df)} rows, {len(user_cols)} user columns')
        print()
        # 按非空行数排序, 显示每列有多少非空值
        col_stats = []
        for c in sorted(user_cols):
            non_null = df[c].apply(lambda v: not is_missing(v)).sum()
            col_stats.append((c, non_null))
        col_stats.sort(key=lambda x: (-x[1], x[0]))
        print(f'  {"Column":<60s} Non-null rows')
        print(f'  {"-"*60} {"-"*13}')
        for col, count in col_stats:
            marker = '' if count > 0 else ' (all NaN!)'
            print(f'  {col:<60s} {count:>5d} / {len(df)}{marker}')

    # ── summary 列 ──
    summary = {k: v for k, v in dict(run.summary).items()
               if not str(k).startswith('_')}
    print()
    print(f'Summary: {len(summary)} keys')
    for k in sorted(summary.keys()):
        v = summary[k]
        v_str = f'{v:.6f}' if isinstance(v, float) else str(v)
        if len(v_str) > 80:
            v_str = v_str[:77] + '...'
        print(f'  {k:<60s} = {v_str}')
    print()


def fetch_table(run, table_prefix, extra_columns=None):
    """
    从 run 的 history 中提取指定前缀的列 + step 指标列 + extra_columns。

    返回:
        history_df : 只包含相关列的 DataFrame
        table_cols : 属于 table_prefix 的列名列表
        step_col   : 检测到的 step 指标列名 (可能为 None)
    """
    rows = list(run.scan_history())
    if not rows:
        return pd.DataFrame(), [], None

    df = pd.DataFrame(rows)

    # 找出属于这张表的列
    table_cols = [c for c in df.columns if c.startswith(table_prefix)]
    if not table_cols:
        raise ValueError(
            f'run {run.name} 的 history 中没有以 "{table_prefix}" 开头的列。'
            f'\n可用列: {sorted(df.columns.tolist())}'
        )

    # step 指标
    step_col = find_step_metric(df.columns.tolist())

    # 要保留的列
    keep = set(table_cols)
    if step_col:
        keep.add(step_col)
    if extra_columns:
        for ec in extra_columns:
            if ec in df.columns:
                keep.add(ec)

    df = df[[c for c in df.columns if c in keep]]

    # 删除全为空的行 (只看 table_cols)
    df = df.dropna(subset=table_cols, how='all').reset_index(drop=True)

    return df, sorted(table_cols), step_col


def rename_columns(df, src_prefix, dst_prefix):
    """把 DataFrame 中 src_prefix 开头的列名替换为 dst_prefix。"""
    if src_prefix == dst_prefix:
        return df, {}
    rename_map = {
        c: c.replace(src_prefix, dst_prefix, 1)
        for c in df.columns if c.startswith(src_prefix)
    }
    return df.rename(columns=rename_map), rename_map


def build_summary(run, src_prefix, dst_prefix):
    """从 run.summary 中提取属于 src_prefix 的 key, 重命名后返回 dict。"""
    result = {}
    for key, value in dict(run.summary).items():
        if str(key).startswith('_'):
            continue
        if str(key).startswith(src_prefix):
            new_key = key.replace(src_prefix, dst_prefix, 1)
            result[new_key] = value
    return result


def preview_task(api, task):
    """预览一个复制任务, 不执行写入。"""
    src_run = find_run_by_name(api, task['src_project'], task['src_run_name'])
    df, table_cols, step_col = fetch_table(
        src_run, task['src_table'], task.get('extra_columns')
    )
    df_renamed, rename_map = rename_columns(df, task['src_table'], task['dst_table'])
    summary = build_summary(src_run, task['src_table'], task['dst_table'])

    # 检查目标 run 是否已存在
    dst_path = f'{ENTITY}/{task["dst_project"]}'
    dst_exists = any(
        r.name == task['dst_run_name']
        for r in api.runs(dst_path)
    )

    print('=' * 72)
    print(f'  src : {ENTITY}/{task["src_project"]} / {task["src_run_name"]} ({src_run.id})')
    print(f'  dst : {ENTITY}/{task["dst_project"]} / {task["dst_run_name"]}')
    print(f'  dst run exists : {dst_exists} {"(will resume)" if dst_exists else "(will create)"}')
    print(f'  src table prefix : {task["src_table"]}')
    print(f'  dst table prefix : {task["dst_table"]}')
    print(f'  table columns ({len(table_cols)}): {table_cols}')
    if rename_map:
        print(f'  rename map: { {k: v for k, v in sorted(rename_map.items())} }')
    print(f'  step metric : {step_col}')
    print(f'  history rows : {len(df_renamed)}')
    print(f'  summary keys ({len(summary)}): {sorted(summary.keys())}')
    print(f'  extra columns : {task.get("extra_columns", [])}')
    print(f'  output columns : {sorted(df_renamed.columns.tolist())}')
    if len(df_renamed) == 0:
        print()
        print('  ⚠️  警告: history rows = 0!')
        print('     该前缀在 history 中全为 NaN, 只存在于 summary 里。')
        print('     请用 inspect_run() 查看源 run 里实际有数据的列名。')
    print()


def execute_task(api, task):
    """执行一个复制任务: 读取源表 → 写入目标 run。"""
    src_run = find_run_by_name(api, task['src_project'], task['src_run_name'])
    df, table_cols, step_col = fetch_table(
        src_run, task['src_table'], task.get('extra_columns')
    )
    df, rename_map = rename_columns(df, task['src_table'], task['dst_table'])
    summary = build_summary(src_run, task['src_table'], task['dst_table'])

    if df.empty and not summary:
        print(f'  ⚠️  跳过 {task["src_run_name"]}: 没有 history 也没有 summary 数据')
        return

    # step_col 重命名后的名字
    if step_col and step_col in rename_map:
        step_col = rename_map[step_col]

    # 检查目标 run 是否已存在
    dst_path = f'{ENTITY}/{task["dst_project"]}'
    existing = [r for r in api.runs(dst_path) if r.name == task['dst_run_name']]

    init_kwargs = {
        'entity': ENTITY,
        'project': task['dst_project'],
        'name': task['dst_run_name'],
        'tags': [f'copied-from:{task["src_project"]}'],
        'notes': (
            f'Table copied from {ENTITY}/{task["src_project"]} '
            f'run={task["src_run_name"]} ({src_run.id}).\n'
            f'src_table={task["src_table"]} -> dst_table={task["dst_table"]}'
        ),
    }

    if existing:
        # resume 已有 run
        dst_run_obj = existing[0]
        init_kwargs['id'] = dst_run_obj.id
        init_kwargs['resume'] = 'must'
        print(f'resuming existing run: {dst_run_obj.id}')
    else:
        print(f'creating new run: {task["dst_run_name"]}')

    new_run = wandb.init(**init_kwargs)
    print(f'  run url: {new_run.url}')

    # 定义 step metric
    if step_col:
        wandb.define_metric(step_col)
        wandb.define_metric('*', step_metric=step_col)
        print(f'  step metric = {step_col}')

    # 上传 history
    if not df.empty:
        for _, row in tqdm(df.iterrows(), total=len(df), desc='  uploading'):
            log_dict = {
                col: val for col, val in row.items()
                if not is_missing(val)
            }
            if log_dict:
                wandb.log(log_dict)
        print(f'  uploaded {len(df)} history rows')
    else:
        print('  ⚠️  no history rows to upload (data only in summary)')

    # 写入 summary
    for key, value in summary.items():
        wandb.run.summary[key] = value
    if summary:
        print(f'  wrote {len(summary)} summary keys')

    wandb.finish()
    print(f'  finished: {task["dst_run_name"]}')
    print()

In [11]:
ENTITY = 'kejian-zhao-tsinghua-university'

TASKS = [
    {
        'src_project': 'work_1201',
        'src_run_name': 'ft_ir101_s3_full_12-13_0',
        'src_table': 'val/IJBC_gt_aligned/Norm:False_Det:True_tpr_at_fpr_0.0001',
        'dst_project': 'eval_log',
        'dst_run_name': 's3_12_13',
        'dst_table': 'summary/IJBC_TPR@FPR0.01',
        'extra_columns': [],
    },
    # 可以继续添加更多任务...
]

# ── 先用 inspect_run 看看源 run 里有哪些列 ──
# 确认 src_table 前缀填对了再跑 preview/execute
api = wandb.Api()
inspect_run(api, 'work_1201', 'ft_ir101_s3_full_12-13_0')

Run: ft_ir101_s3_full_12-13_0 (h7xttgzs)
URL: https://wandb.ai/kejian-zhao-tsinghua-university/work_1201/runs/h7xttgzs

History: 9000 rows, 76 user columns

  Column                                                       Non-null rows
  ------------------------------------------------------------ -------------
  trainer/global_step                                           9000 / 9000
  epoch                                                         8973 / 9000
  n_images_seen                                                 8973 / 9000
  step                                                          8973 / 9000
  train/adaface_batch_mean                                      8811 / 9000
  train/adaface_batch_std                                       8811 / 9000
  train/grad_norm_backbone                                      8811 / 9000
  train/grad_norm_classifier                                    8811 / 9000
  train/loss                                                    8811 / 9000
  tra

In [12]:
# ── Step 1: 预览 ─────────────────────────────────────────────────
# 先看看将要复制的内容, 确认无误后再执行下一个 cell。
api = wandb.Api()
for task in TASKS:
    preview_task(api, task)

  src : kejian-zhao-tsinghua-university/work_1201 / ft_ir101_s3_full_12-13_0 (h7xttgzs)
  dst : kejian-zhao-tsinghua-university/eval_log / s3_12_13
  dst run exists : True (will resume)
  src table prefix : val/IJBC_gt_aligned/Norm:False_Det:True_tpr_at_fpr_0.0001
  dst table prefix : summary/IJBC_TPR@FPR0.01
  table columns (1): ['val/IJBC_gt_aligned/Norm:False_Det:True_tpr_at_fpr_0.0001']
  rename map: {'val/IJBC_gt_aligned/Norm:False_Det:True_tpr_at_fpr_0.0001': 'summary/IJBC_TPR@FPR0.01'}
  step metric : trainer/global_step
  history rows : 27
  summary keys (1): ['summary/IJBC_TPR@FPR0.01']
  extra columns : []
  output columns : ['summary/IJBC_TPR@FPR0.01', 'trainer/global_step']



In [13]:
# ── Step 2: 执行 ─────────────────────────────────────────────────
# 确认 preview 没问题后, 执行这个 cell 真正写入。
api = wandb.Api()
for task in TASKS:
    execute_task(api, task)

resuming existing run: w00uwrr5


  run url: https://wandb.ai/kejian-zhao-tsinghua-university/eval_log/runs/w00uwrr5
  step metric = trainer/global_step


  uploading:   0%|          | 0/27 [00:00<?, ?it/s]

  uploaded 27 history rows
  wrote 1 summary keys


summary/IJBC_TPR@FPR0.01,█▁▁█▁▁█▁▁█▁▁█▁▁█▁▁█▁▁█▁▁█▁▁
trainer/global_step,▁▅█▁▅█▁▅█▁▅█▁▅█▁▅█▁▅█▁▅█▁▅█
epoch,25
n_images_seen,0
step,0
summary/IJBC_TPR@FPR0.01,93.55218
summary/ijbc_001_tpir_at_far_1e-05,90.70655
summary/ijbc_001_tpir_at_far_1e-06,72.77777
summary/ijbc_001_tpir_at_far_1e-07,11.48847
summary/ijbc_001_tpir_at_far_1e-08,0.59401
summary/ijbc_001_tpir_at_far_1e-09,0.59401


  finished: s3_12_13

